<div>
 
  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>
 
  <hr>
 
  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-04-27</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>
 
  <hr>
 
  <div style="text-align:center;">
    <h1><b>Ejercicio 3 — Modelo Vectorial y TF-IDF</b></h1>
  </div>
 
</div>

Implementación del modelo vectorial para recuperación de información en dos corpus de distinta escala:

- **Parte 1 — Corpus turismo:** implementación densa desde cero con NumPy (500 documentos)
- **Parte 2 — Corpus Gutenberg 1000:** implementación con matrices sparse usando sklearn (1000 libros)

El objetivo es entender la mecánica del modelo vectorial y por qué no escala a corpus grandes sin estructuras eficientes.

## Imports

In [2]:
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append("../")
import libreria as ir

## Función compartida: tokenización

Ambas partes usan la misma función de tokenización. Se elimina el **BOM** (`\ufeff`),
un carácter invisible que algunos archivos UTF-8 incluyen al inicio y que contaminaría el vocabulario.

In [3]:
def tokenize(text: str) -> list:
    """Convierte un texto en lista de tokens normalizados."""
    return text.lower().replace(".", "").replace(",", "").replace("\ufeff", "").split()

---
# Parte 1 — Corpus turismo: implementación desde cero con NumPy

Corpus pequeño (500 documentos) ideal para construir el modelo vectorial paso a paso
y entender la mecánica detrás de cada componente.

## 1. Carga del corpus

El corpus `01_corpus_turismo_500.txt` contiene 500 documentos breves sobre turismo en Ecuador,
uno por línea. Se carga como string completo y se divide por saltos de línea.

In [4]:
# Cargar el archivo como string y separar por líneas
corpus_turismo_raw = ir.load_file('01_corpus_turismo_500.txt')
docs_turismo = [doc for doc in corpus_turismo_raw.split('\n') if doc]

print(f"Documentos cargados: {len(docs_turismo)}")
print(f"Ejemplo — doc[0]: {docs_turismo[0]}")

Documentos cargados: 500
Ejemplo — doc[0]: Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.


## 2. Tokenización

In [5]:
docs_tok_turismo = [tokenize(doc) for doc in docs_turismo]

print(f"Ejemplo — doc[0] tokenizado: {docs_tok_turismo[0]}")

Ejemplo — doc[0] tokenizado: ['otavalo', 'es', 'conocido', 'por', 'su', 'mercado', 'indígena', 'y', 'su', 'artesanía', 'perfecto', 'para', 'rafting']


## 3. Vocabulario

El vocabulario es el conjunto de todos los términos únicos del corpus.
Se ordena alfabéticamente para garantizar un orden fijo y reproducible —
cada posición en el vocabulario corresponde a una columna de la matriz.

In [6]:
# Construir el conjunto de términos únicos y ordenarlo
vocab_turismo = set()
for doc in docs_tok_turismo:
    vocab_turismo.update(doc)

vocab_turismo = sorted(vocab_turismo)  # lista ordenada para índices consistentes

N_docs_turismo  = len(docs_turismo)
N_vocab_turismo = len(vocab_turismo)

print(f"Documentos : {N_docs_turismo}")
print(f"Vocabulario: {N_vocab_turismo} términos")

Documentos : 500
Vocabulario: 118 términos


## 4. Term Frequency (TF)

**TF(t, d)** mide qué tan frecuente es el término `t` dentro del documento `d`.
Se usa la versión normalizada para evitar que documentos más largos tengan ventaja:

$$TF(t, d) = \frac{\text{conteo de } t \text{ en } d}{\text{total de tokens en } d}$$

In [7]:
def tf(termino: str, doc: list) -> float:
    """Calcula la frecuencia normalizada de un término en un documento."""
    return len([token for token in doc if token == termino]) / len(doc)


# Construir la matriz TF de shape (N_docs x N_vocab)
matriz_tf_turismo = np.zeros((N_docs_turismo, N_vocab_turismo))
for i, doc in enumerate(docs_tok_turismo):
    for j, term in enumerate(vocab_turismo):
        matriz_tf_turismo[i, j] = tf(term, doc)

print(f"Shape de la matriz TF: {matriz_tf_turismo.shape}")

Shape de la matriz TF: (500, 118)


## 5. Inverse Document Frequency (IDF)

**IDF(t)** mide qué tan raro es un término en el corpus: términos que aparecen en pocos
documentos reciben mayor peso y son más discriminativos para la búsqueda.
El logaritmo suaviza el crecimiento para que diferencias grandes en `df` no dominen:

$$IDF(t) = \log\left(\frac{N}{df(t)}\right)$$

donde `df(t)` es el número de documentos que contienen el término `t`.

In [8]:
def idf(termino: str, docs: list) -> float:
    """Calcula el IDF de un término sobre el corpus."""
    df = sum([termino in doc for doc in docs])  # nro de docs que contienen el término
    return np.log(len(docs) / df)


# Calcular el vector IDF de shape (N_vocab,) — un valor por término
idf_vector_turismo = np.zeros(N_vocab_turismo)
for i, palabra in enumerate(vocab_turismo):
    idf_vector_turismo[i] = idf(palabra, docs_tok_turismo)

print(f"Shape del vector IDF: {idf_vector_turismo.shape}")

Shape del vector IDF: (118,)


## 6. Matriz TF-IDF

La matriz TF-IDF se obtiene multiplicando elemento a elemento la matriz TF por el vector IDF.
Gracias al broadcasting de NumPy, el vector `(N_vocab,)` se aplica automáticamente
a cada fila de la matriz `(N_docs, N_vocab)`.

In [9]:
# Broadcasting: cada fila (documento) se multiplica por el vector IDF
matriz_tfidf_turismo = matriz_tf_turismo * idf_vector_turismo

print(f"Shape de la matriz TF-IDF: {matriz_tfidf_turismo.shape}")

Shape de la matriz TF-IDF: (500, 118)


## 7. Normalización

Se normaliza cada vector de documento a norma 1 para que la similitud coseno
compare solo la **dirección** del vector, no su magnitud.
Esto evita que documentos más largos tengan ventaja por valores más grandes.

In [10]:
# Norma de cada fila → reshape a (N_docs, 1) para broadcasting correcto
normas_turismo        = np.linalg.norm(matriz_tfidf_turismo, axis=1).reshape(-1, 1)
matriz_norm_turismo   = matriz_tfidf_turismo / normas_turismo

print(f"Shape de la matriz normalizada: {matriz_norm_turismo.shape}")

Shape de la matriz normalizada: (500, 118)


## 8. Búsqueda por similitud coseno

Para buscar, la consulta se convierte al mismo espacio vectorial que los documentos
(tokenizar → calcular TF-IDF → normalizar). Luego se calcula el **producto punto**
contra cada fila de la matriz normalizada, que equivale a la similitud coseno
cuando ambos vectores están normalizados.

In [11]:
def vectorizar(tokens: list, vocabulario: list, docs: list) -> np.ndarray:
    """Convierte una lista de tokens en su vector TF-IDF normalizado."""
    vector = np.zeros(len(vocabulario))
    for i, term in enumerate(vocabulario):
        vector[i] = tf(term, tokens) * idf(term, docs)
    norma = np.linalg.norm(vector)
    return vector / norma

In [12]:
def buscar_turismo(consulta: str, top_k: int = 5) -> None:
    """Busca documentos relevantes en el corpus de turismo usando similitud coseno."""
    tokens = tokenize(consulta)
    vector_consulta = vectorizar(tokens, vocab_turismo, docs_tok_turismo)

    # Producto punto = similitud coseno (vectores ya normalizados)
    scores = np.dot(matriz_norm_turismo, vector_consulta)

    # Ordenar de mayor a menor y tomar los top_k
    rank = np.argsort(scores)[-top_k:][::-1]
    for i in rank:
        print(f"{scores[i]:.4f} — {docs_turismo[i]}")

## 9. Ejemplos

In [13]:
buscar_turismo("playas y naturaleza en Ecuador")

0.4378 — Vilcabamba atrae visitantes interesados en longevidad y naturaleza
0.4378 — Vilcabamba atrae visitantes interesados en longevidad y naturaleza
0.4378 — Vilcabamba atrae visitantes interesados en longevidad y naturaleza
0.4158 — Vilcabamba atrae visitantes interesados en longevidad y naturaleza Ideal para el próximo feriado.
0.4158 — Vilcabamba atrae visitantes interesados en longevidad y naturaleza Ideal para el próximo feriado.


In [14]:
buscar_turismo("volcán y senderismo")

0.5391 — El volcán Cotopaxi es un destino de senderismo popular
0.5391 — El volcán Cotopaxi es un destino de senderismo popular
0.5391 — El volcán Cotopaxi es un destino de senderismo popular
0.5391 — El volcán Cotopaxi es un destino de senderismo popular
0.5391 — El volcán Cotopaxi es un destino de senderismo popular


In [15]:
buscar_turismo("gastronomía colonial")

0.5349 — Cuenca deslumbra con su arquitectura colonial y gastronomía
0.5349 — Cuenca deslumbra con su arquitectura colonial y gastronomía
0.5349 — Cuenca deslumbra con su arquitectura colonial y gastronomía
0.5349 — Cuenca deslumbra con su arquitectura colonial y gastronomía
0.5349 — Cuenca deslumbra con su arquitectura colonial y gastronomía


---
# Parte 2 — Corpus Gutenberg 1000: matrices sparse con sklearn

El mismo modelo vectorial aplicado a 1000 libros completos del Proyecto Gutenberg (~429 MB).
El objetivo es entender por qué la implementación densa no escala y cómo sklearn lo resuelve.

## 1. Carga del corpus

`load_corpus()` retorna un dict `{nombre_archivo: contenido}` — a diferencia del corpus de turismo,
aquí cada documento es un libro completo, no una línea.

In [16]:
corpus_gutenberg = ir.load_corpus('../corpus')

print(f"Libros cargados: {len(corpus_gutenberg)}")

Corpus cargado: 1000 archivo(s) desde '../corpus'
Libros cargados: 1000


## 2. Tokenización

Se usa la misma función `tokenize()` definida al inicio. El corpus es un dict,
por lo que se tokeniza sobre los valores (contenidos de cada libro).

In [17]:
# Corpus es un dict → se tokeniza sobre los valores (contenidos)
docs_tok_gutenberg = [tokenize(doc) for doc in corpus_gutenberg.values()]

print(f"Ejemplo — doc[0] (primeros 10 tokens): {docs_tok_gutenberg[0][:10]}")

Ejemplo — doc[0] (primeros 10 tokens): ['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'house', 'of', 'the', 'whispering']


## 3. Vocabulario

Con 1000 libros, el vocabulario explota. La tokenización básica con `split()` trata
`"hello"`, `"hello!"` y `"Hello"` como términos distintos — a esta escala eso multiplica
el vocabulario de forma problemática.

In [18]:
vocab_gutenberg = set()
for doc in docs_tok_gutenberg:
    vocab_gutenberg.update(doc)

vocab_gutenberg = sorted(vocab_gutenberg)

N_docs_gutenberg  = len(corpus_gutenberg)
N_vocab_gutenberg = len(vocab_gutenberg)

print(f"Documentos : {N_docs_gutenberg}")
print(f"Vocabulario: {N_vocab_gutenberg} términos")

Documentos : 1000
Vocabulario: 1703151 términos


## 4. El problema de escala — matriz densa imposible

Con los valores obtenidos, intentar construir la matriz TF densa equivale a alojar en RAM:

$$1000 \times 1{,}703{,}151 \times 8 \text{ bytes} \approx 13.6 \text{ GB}$$

El resultado es un `MemoryError` inmediato. El problema no es solo la memoria —
la mayoría de esas celdas serían **cero** (un libro no usa ni el 1% del vocabulario total).
Guardar todos esos ceros es un desperdicio brutal.

```
MemoryError: Unable to allocate 12.7 GiB for an array
with shape (1000, 1703151) and data type float64
```

La solución es usar **matrices sparse**: estructuras que solo almacenan los valores distintos de cero.

## 5. Solución: TF-IDF con sklearn y matrices sparse

### Preparación del corpus

`TfidfVectorizer` espera strings, no listas de tokens.
Se reconstruye el texto limpio uniendo los tokens ya tokenizados — así se preserva
la limpieza del BOM sin reprocesar los archivos originales.

In [19]:
# Reconstruir strings limpios desde los tokens
corpus_limpio = [' '.join(doc) for doc in docs_tok_gutenberg]

### Construcción de la matriz TF-IDF

`fit_transform()` hace en una sola llamada lo que en la Parte 1 tomó varios pasos:
tokenización, construcción del vocabulario, cálculo de TF-IDF y normalización.

El resultado es una **CSR matrix** (Compressed Sparse Row) — solo almacena los valores
distintos de cero, haciendo viable la operación que antes causaba `MemoryError`.

> **Nota:** sklearn usa `log(N/df) + 1` en lugar de `log(N/df)`, así que los valores
> no son idénticos a la implementación manual, pero el comportamiento es equivalente.

In [20]:
vectorizer_gutenberg = TfidfVectorizer()
matriz_tfidf_gutenberg = vectorizer_gutenberg.fit_transform(corpus_limpio)

print(f"Shape de la matriz: {matriz_tfidf_gutenberg.shape}")
print(f"Elementos almacenados (no-cero): {matriz_tfidf_gutenberg.nnz:,}")
print(f"Densidad: {matriz_tfidf_gutenberg.nnz / (matriz_tfidf_gutenberg.shape[0] * matriz_tfidf_gutenberg.shape[1]):.4%}")

Shape de la matriz: (1000, 675909)
Elementos almacenados (no-cero): 6,991,402
Densidad: 1.0344%


## 6. Búsqueda por similitud coseno

`vectorizer.transform()` convierte la consulta al mismo espacio vectorial usando el vocabulario
ya aprendido. El producto punto `@` entre la matriz sparse y el vector de consulta
equivale a la similitud coseno (los vectores ya están normalizados por sklearn).

In [21]:
titulos_gutenberg = list(corpus_gutenberg.keys())  # orden consistente con la matriz


def buscar_gutenberg(consulta: str, top_k: int = 5) -> None:
    """Busca libros relevantes en el corpus Gutenberg usando similitud coseno sobre TF-IDF."""
    vector_consulta = vectorizer_gutenberg.transform([consulta])

    # Producto punto = similitud coseno (vectores ya normalizados por sklearn)
    scores = (matriz_tfidf_gutenberg @ vector_consulta.T).toarray().flatten()

    rank = np.argsort(scores)[-top_k:][::-1]
    for i in rank:
        print(f"{scores[i]:.4f} — {titulos_gutenberg[i]}")

## 7. Ejemplos

In [22]:
buscar_gutenberg("love and adventure")

0.2427 — 76_Adventures of Huckleberry Finn.txt
0.2399 — 53545_Canadian Battlefields, and Other Poems.txt
0.2371 — 3228_Poems of Progress and New Thought Pastels.txt
0.2369 — 10380_Bible Stories and Religious Classics.txt
0.2364 — 51252_The Book of the Thousand Nights and a Night Vol 01.txt


In [23]:
buscar_gutenberg("war and politics")

0.2142 — 76_Adventures of Huckleberry Finn.txt
0.2106 — 53545_Canadian Battlefields, and Other Poems.txt
0.2083 — 10380_Bible Stories and Religious Classics.txt
0.2045 — 51252_The Book of the Thousand Nights and a Night Vol 01.txt
0.2041 — 4196_Diary of Samuel Pepys — Volume 71 January 1668-69.txt


In [24]:
buscar_gutenberg("mystery and detective")

0.1691 — 19929_Cad Metti, The Female Detective Strategist; Or, Dudie Dunne.txt
0.1604 — 76_Adventures of Huckleberry Finn.txt
0.1565 — 53545_Canadian Battlefields, and Other Poems.txt
0.1548 — 10380_Bible Stories and Religious Classics.txt
0.1531 — 4196_Diary of Samuel Pepys — Volume 71 January 1668-69.txt
